In [ ]:
'''
Script that takes the paths for the original spreadsheet, the folder with the train set and the folder with the validation set
and produces a new spreadsheet with additional columns with properties of the images comprising the dataset.

For a preliminary analysis of a dataset intended for object detection it's important to focus on properties that provide insights into the dataset's characteristics and variability, without delving too deeply into complex feature extractions.
We look into the following properties:
- Image Dimensions: Width and height of each image. This helps to understand the size distribution of the images and whether any resizing or normalization is needed.
- Aspect Ratio: The ratio of width to height. This can be important for understanding how much variation there is in the shapes of the images.
- Color Distribution: Basic color analysis based on the average intensity per channel (Red, Green, Blue).
- Contrast and Brightness: Basic statistics on the brightness and contrast of the images can be useful to assess the need for pre-processing steps like contrast normalization.
- Presence of Blurriness assessed using the variance of the Laplacian.
'''

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
from datetime import datetime

# Function to calculate image properties
def get_image_properties(image_path):
    image = cv2.imread(image_path)
    height, width, channels = image.shape

    # Calculating aspect ratio
    aspect_ratio = width / height if height != 0 else 0

    # Calculating average intensity per channel (Red, Green, Blue)
    avg_intensity_per_channel = np.mean(image, axis=(0, 1))
    avg_red_intensity = avg_intensity_per_channel[0]
    avg_green_intensity = avg_intensity_per_channel[1]
    avg_blue_intensity = avg_intensity_per_channel[2]

    # Calculating brightness and contrast
    brightness = np.mean(avg_intensity_per_channel)
    contrast = np.std(image)

    # Blurriness
    laplacian_var = cv2.Laplacian(image, cv2.CV_64F).var()

    return width, height, aspect_ratio, avg_red_intensity, avg_green_intensity, avg_blue_intensity, brightness, contrast, laplacian_var

# Main function to update the spreadsheet with image properties
def update_spreadsheet_with_image_properties():
    
    # Load the spreadsheet
    root = tk.Tk()
    root.withdraw()  # Hide the main window

    # Open file dialog to select the spreadsheet file
    spreadsheet_path = filedialog.askopenfilename(
        title="Select the Spreadsheet File",
        filetypes=[("Excel files", "*.xlsx")]
    )
    
    # Open directory dialogs to select train and validation folders
    train_folder_path = filedialog.askdirectory(title="Select Train Set Folder")
    validation_folder_path = filedialog.askdirectory(title="Select Validation Set Folder")
    
    if not (spreadsheet_path and train_folder_path and validation_folder_path):
        return "File paths must be provided."

    # Read the spreadsheet
    spreadsheet = pd.ExcelFile(spreadsheet_path)

    # Process Train Set
    train_df = pd.read_excel(spreadsheet, 'train_set')
    train_df[['Width', 'Height', 'Aspect Ratio', 'Avg Red Intensity', 'Avg Green Intensity', 
              'Avg Blue Intensity', 'Brightness', 'Contrast', 'Blurriness']] = None

    for idx, row in train_df.iterrows():
        patient_folder = os.path.join(train_folder_path, row['PATIENT_ID'])
        polyp_folder = os.path.join(patient_folder, row['POLYP_ID'])
        image_path = os.path.join(polyp_folder, row['IMG_FILE_PREFIX'] + ".full.png")
        if os.path.exists(image_path):
            properties = get_image_properties(image_path)
            train_df.loc[idx, ['Width', 'Height', 'Aspect Ratio', 'Avg Red Intensity', 
                           'Avg Green Intensity', 'Avg Blue Intensity', 'Brightness', 
                           'Contrast', 'Blurriness']] = properties
    
    # Process Validation Set
    validation_df = pd.read_excel(spreadsheet, 'validation_set')
    validation_df[['Width', 'Height', 'Aspect Ratio', 'Avg Red Intensity', 'Avg Green Intensity', 
                   'Avg Blue Intensity', 'Brightness', 'Contrast', 'Blurriness']] = None

    for idx, row in validation_df.iterrows():
        patient_folder = os.path.join(validation_folder_path, row['PATIENT_ID'])
        image_id = row['POLYP/IMAGE_ID']
        image_path = os.path.join(patient_folder, image_id + ".png")

        if os.path.exists(image_path):
            properties = get_image_properties(image_path)
            validation_df.loc[idx, ['Width', 'Height', 'Aspect Ratio', 'Avg Red Intensity', 
                           'Avg Green Intensity', 'Avg Blue Intensity', 'Brightness', 
                           'Contrast', 'Blurriness']] = properties
    # Save the updated spreadsheets
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    new_spreadsheet_path = spreadsheet_path.replace('.xlsx', f'_{timestamp}.xlsx')
    with pd.ExcelWriter(new_spreadsheet_path) as writer:
        train_df.to_excel(writer, sheet_name='train_set', index=False)
        validation_df.to_excel(writer, sheet_name='validation_set', index=False)
    return f"Updated spreadsheet saved as {new_spreadsheet_path}"


# Call the function with placeholder paths (to be replaced with actual paths)
update_spreadsheet_result = update_spreadsheet_with_image_properties()
update_spreadsheet_result


'Updated spreadsheet saved as /Users/irconde/Downloads/Databases/POLAR/polar_public/polar_annotations_20231125102933.xlsx'